# Pipeline Hán - Việt: Tự Động Dóng Hàng & Huấn Luyện Máy Dịch trên Kaggle GPU (Zero-Setup)

Notebook này tự động thực hiện toàn bộ quy trình từ **Dóng hàng câu** đến **Huấn luyện máy dịch** bằng cách clone trực tiếp mã nguồn và dữ liệu từ GitHub của bạn. Bạn không cần tải dữ liệu hay tạo Dataset thủ công trên Kaggle.

## Hướng dẫn chạy nhanh:
1. Hãy push toàn bộ nhánh `features/mapping-translation` hiện tại ở local lên GitHub của bạn.
2. Tạo một Kaggle Notebook mới, thiết lập **Accelerator** là **GPU T4 x2** hoặc **GPU P100** ở bảng cài đặt bên phải.
3. Thay thế URL GitHub của bạn vào ô code bên dưới và nhấn **Run All**.

## Bước 1: Clone/Update Repository từ GitHub và chuyển thư mục làm việc

In [ ]:
# THAY THẾ URL GITHUB REPO CỦA BẠN VÀO ĐÂY
GITHUB_REPO_URL = "https://github.com/YOUR_GITHUB_USERNAME/SinoNom-NLP.git"

import os
repo_name = GITHUB_REPO_URL.split("/")[-1].replace(".git", "")

if not os.path.exists(repo_name):
    print(f"Cloning repository from {GITHUB_REPO_URL}...")
    # Clone nhánh features/mapping-translation chứa toàn bộ dữ liệu và code mới
    !git clone -b features/mapping-translation {GITHUB_REPO_URL}
else:
    print("Repository already exists. Fetching and updating to the latest commit...")
    %cd {repo_name}
    !git fetch --all
    !git reset --hard origin/features/mapping-translation
    %cd ..

# Chuyển thư mục làm việc vào trong repo
%cd {repo_name}
!ls -la

## Bước 2: Cài đặt các thư viện cần thiết

In [ ]:
# Cài đặt các thư viện phục vụ cho việc dóng hàng và huấn luyện dịch máy
!pip install -q sentence-transformers pandas openpyxl
!pip install -q transformers[torch] datasets evaluate sacrebleu accelerate tensorboard
!pip install -q bertalign simalign bitsandbytes

## Bước 3: Phase 1 — Chạy Ensemble Alignment (Không dùng Qwen)

Chúng ta sẽ chạy mô hình dóng hàng kết hợp (Ensemble Aligner) gồm LaBSE, Vecalign, BERTAlign, và SimAlign để tạo ra kết quả dóng hàng ban đầu mà chưa lọc qua Qwen.

**Lưu ý về Chế độ Thử nghiệm (Dev Mode):**
* Lệnh dưới đây mặc định bật cờ `--dev` để chạy thử nghiệm nhanh trên **Quyển 1**.
* Để chạy dóng hàng trên toàn bộ 17 quyển, hãy xóa cờ `--dev` đi.

In [ ]:
# Chạy Phase 1: Ensemble Alignment
!python run_mapping.py \
    --aligner ensemble \
    --sino_dir dataset/MAPPING/sino_extract \
    --viet_dir dataset/MAPPING/vietnam_extract/csv \
    --output_dir output \
    --work_code HVB_001 \
    --device cuda \
    --dev

### Kiểm tra trực quan kết quả Phase 1

In ra 20 dòng đầu tiên của kết quả dóng hàng sau Phase 1.

In [ ]:
import glob
import pandas as pd
from IPython.display import display

tsv_files = glob.glob("output/HVB_001/**/*.tsv", recursive=True)
if tsv_files:
    sample_file = sorted(tsv_files)[0]
    print(f"Kết quả Phase 1 (File: {sample_file}):\n")
    df_sample = pd.read_csv(sample_file, sep="\t")
    pd.set_option('display.max_colwidth', None)
    display(df_sample.head(20))
else:
    print("Chưa tìm thấy kết quả dóng hàng.")

### Tải về kết quả Phase 1 (Download Phase 1 Files)

Chạy ô code dưới đây để nén toàn bộ thư mục `output` thành file zip. Bạn có thể tải file `output_phase1.zip` trực tiếp từ cột bên phải của giao diện Kaggle.

In [ ]:
# Xóa file zip cũ nếu có và nén kết quả
!rm -f output_phase1.zip
!zip -q -r output_phase1.zip output/
print("Đã nén xong! Hãy tải file 'output_phase1.zip' ở cột bên phải Kaggle để kiểm tra.")

## Bước 3.5: Phase 2 — Chạy Qwen LLM Verification (Lọc & Tách lỗi)

Sau khi đã kiểm tra kết quả Phase 1, ta tiến hành chạy Phase 2 bằng cách thêm cờ `--qwen`. 
Qwen2.5-7B-Instruct sẽ được nạp offline ở chế độ 4-bit, tự động duyệt qua các câu dóng lệch và tách rời chúng ra.

In [ ]:
# Chạy Phase 2: Qwen Verification
# Lưu ý: Phải dùng GPU để chạy được Qwen
!python run_mapping.py \
    --aligner ensemble \
    --qwen \
    --sino_dir dataset/MAPPING/sino_extract \
    --viet_dir dataset/MAPPING/vietnam_extract/csv \
    --output_dir output \
    --work_code HVB_001 \
    --device cuda \
    --dev

### Kiểm tra trực quan kết quả sau khi qua bộ lọc Qwen

In ra 20 dòng đầu tiên của kết quả để xem các cặp lỗi đã được Qwen tách ra sạch sẽ chưa.

In [ ]:
tsv_files = glob.glob("output/HVB_001/**/*.tsv", recursive=True)
if tsv_files:
    sample_file = sorted(tsv_files)[0]
    print(f"Kết quả sau khi qua lọc Qwen (File: {sample_file}):\n")
    df_sample = pd.read_csv(sample_file, sep="\t")
    pd.set_option('display.max_colwidth', None)
    display(df_sample.head(20))
else:
    print("Chưa tìm thấy kết quả dóng hàng.")

### Tải về kết quả sau cùng (Download Final Phase 2 Files)

Click tải file `output_final.zip` ở cột bên phải.

In [ ]:
!rm -f output_final.zip
!zip -q -r output_final.zip output/
print("Đã nén xong! Hãy tải file 'output_final.zip' ở cột bên phải Kaggle để gửi lại cho mình kiểm tra.")

## Bước 4: Chuẩn bị dữ liệu huấn luyện Dịch máy

Ta chạy script `prepare_data.py` để gộp toàn bộ các file tsv song song đã được dóng hàng thành công, chia tập Train/Val (90/10) và lưu thành định dạng JSONLines chuẩn.

In [ ]:
# Tạo train.json và val.json từ kết quả dóng hàng
!python scripts/prepare_data.py

# Kiểm tra file đầu ra
!ls -la output/translation_dataset

## Bước 5: Huấn luyện mô hình dịch (Fine-tuning)

Tải script huấn luyện chuẩn từ Hugging Face và tiến hành tinh chỉnh mô hình dịch Trung-Việt.

**Lưu ý:** `--overwrite_output_dir` được viết dưới dạng cờ (flag) boolean chuẩn của HfArgumentParser.

In [ ]:
# Tải script huấn luyện chính thức
!wget -q https://raw.githubusercontent.com/huggingface/transformers/main/examples/pytorch/translation/run_translation.py

# Chạy tinh chỉnh mô hình Helsinki-NLP/opus-mt-zh-vi trên GPU
!python run_translation.py \
    --model_name_or_path Helsinki-NLP/opus-mt-zh-vi \
    --source_lang zh \
    --target_lang vi \
    --train_file output/translation_dataset/train.json \
    --validation_file output/translation_dataset/val.json \
    --output_dir ./han_viet_translation_model \
    --per_device_train_batch_size 16 \
    --per_device_eval_batch_size 16 \
    --overwrite_output_dir \
    --do_train \
    --do_eval \
    --num_train_epochs 5 \
    --learning_rate 2e-5 \
    --weight_decay 0.01 \
    --predict_with_generate \
    --eval_strategy epoch \
    --save_strategy epoch

## Bước 6: Kiểm tra khả năng dịch của Mô hình

Dịch thử câu mẫu Hán cổ bằng checkpoint vừa train.

In [ ]:
import os
from transformers import MarianMTModel, MarianTokenizer

# Sử dụng đường dẫn tuyệt đối để tránh lỗi xác thực Repo ID của Hugging Face
model_path = os.path.abspath("./han_viet_translation_model")

if os.path.exists(model_path) and os.path.exists(os.path.join(model_path, "config.json")):
    print(f"Loading model from: {model_path}")
    tokenizer = MarianTokenizer.from_pretrained(model_path)
    model = MarianMTModel.from_pretrained(model_path)

    def translate(text):
        inputs = tokenizer(text, return_tensors="pt", padding=True, truncation=True)
        translated = model.generate(**inputs)
        return tokenizer.decode(translated[0], skip_special_tokens=True)

    # Test dịch câu mẫu
    sample_han = "大南一統志卷之二承天府上"
    print(f"Hán: {sample_han}")
    print(f"Dịch: {translate(sample_han)}")
else:
    print(f"[Lưu ý] Không tìm thấy thư mục mô hình hoặc config.json tại '{model_path}'.")
    print("Vui lòng kiểm tra xem bước huấn luyện (Bước 5) đã chạy hoàn thành và lưu kết quả thành công chưa.")